# Notebook 1 — Exploratory Data Analysis (EDA)
**Credit Card Fraud Detection | IS525E Data Science for Business**

This notebook explores the dataset to understand:
- Class distribution (fraud vs. legitimate)
- Transaction amount and time patterns
- Feature distributions and correlations

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## 1. Load Dataset

In [ ]:
df = pd.read_csv('../data/creditcard.csv')
print(f'Shape: {df.shape}')
df.head()

## 2. Basic Info & Missing Values

In [ ]:
df.info()
print('\nMissing values per column:')
print(df.isnull().sum())

In [ ]:
df.describe()

## 3. Class Distribution

In [ ]:
class_counts = df['Class'].value_counts()
fraud_pct = class_counts[1] / len(df) * 100
print(f'Legitimate transactions: {class_counts[0]:,}')
print(f'Fraudulent transactions:  {class_counts[1]:,}  ({fraud_pct:.2f}%)')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(['Legitimate', 'Fraud'], class_counts, color=['steelblue', 'tomato'])
axes[0].set_title('Transaction Count by Class')
axes[0].set_ylabel('Count')
axes[1].pie(class_counts, labels=['Legitimate', 'Fraud'], autopct='%1.2f%%', colors=['steelblue', 'tomato'])
axes[1].set_title('Class Proportion')
plt.tight_layout()
plt.savefig('../reports/class_distribution.png', dpi=150)
plt.show()

## 4. Transaction Amount Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for i, (label, color) in enumerate([(0, 'steelblue'), (1, 'tomato')]):
    axes[i].hist(df[df['Class'] == label]['Amount'], bins=50, color=color, edgecolor='white')
    axes[i].set_title(f'Amount Distribution — {"Legitimate" if label == 0 else "Fraud"}')
    axes[i].set_xlabel('Amount (€)')
    axes[i].set_ylabel('Count')
plt.tight_layout()
plt.savefig('../reports/amount_distribution.png', dpi=150)
plt.show()

print('Fraud amount stats:')
print(df[df['Class'] == 1]['Amount'].describe())

## 5. Transactions Over Time

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6))
for i, (label, color, title) in enumerate([
    (0, 'steelblue', 'Legitimate Transactions Over Time'),
    (1, 'tomato', 'Fraudulent Transactions Over Time')
]):
    axes[i].scatter(df[df['Class'] == label]['Time'], df[df['Class'] == label]['Amount'],
                    alpha=0.3, s=5, color=color)
    axes[i].set_title(title)
    axes[i].set_xlabel('Time (seconds)')
    axes[i].set_ylabel('Amount (€)')
plt.tight_layout()
plt.savefig('../reports/transactions_over_time.png', dpi=150)
plt.show()

## 6. Feature Correlation Heatmap

In [ ]:
plt.figure(figsize=(18, 14))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0,
            square=True, linewidths=0.5, annot=False)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('../reports/correlation_heatmap.png', dpi=150)
plt.show()

## 7. Top Features Correlated with Fraud

In [ ]:
fraud_corr = df.corr()['Class'].drop('Class').sort_values(key=abs, ascending=False)
print('Top 10 features correlated with Class (fraud):')
print(fraud_corr.head(10))

plt.figure(figsize=(10, 5))
fraud_corr.head(10).plot(kind='bar', color=['tomato' if x < 0 else 'steelblue' for x in fraud_corr.head(10)])
plt.title('Top 10 Features by Correlation with Fraud')
plt.ylabel('Correlation Coefficient')
plt.axhline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.savefig('../reports/top_feature_correlations.png', dpi=150)
plt.show()

## Key EDA Findings

- **Severe class imbalance**: only ~0.17% of transactions are fraudulent — models must account for this
- **Fraud amounts**: tend to be smaller than typical transactions (fraudsters avoid large unusual amounts)
- **Top predictive features**: V17, V14, V12, V10, V16 show the strongest correlation with fraud
- **No missing values** in the dataset
- **PCA features V1–V28** are already anonymized and scaled; `Time` and `Amount` will need scaling

**Next step:** Notebook 02 — Data Preprocessing